In [2]:
import pandas as pd

In [9]:
data = pd.read_csv('crimedata_2019_2023.csv')
data.head()

,TYPE,YEAR,MONTH,DAY,HOUR,MINUTE,HUNDRED_BLOCK,NEIGHBOURHOOD,X,Y
0,Break and Enter Commercial,2019,3,7,2,6,10XX SITKA SQ,Fairview,490612.9648,5.457110e+06
1,Break and Enter Commercial,2019,8,27,4,12,10XX ALBERNI ST,West End,491004.8164,5.459177e+06
2,Break and Enter Commercial,2021,4,26,4,44,10XX ALBERNI ST,West End,491007.7798,5.459174e+06
3,Break and Enter Commercial,2020,7,28,19,12,10XX ALBERNI ST,West End,491015.9434,5.459166e+06
4,Break and Enter Commercial,2021,11,21,6,33,10XX ALBERNI ST,West End,491015.9434,5.459166e+06


In [ ]:
# --- Feature engineering for dashboard-ready signals ---

data['incident_datetime'] = pd.to_datetime(
    data[['YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE']]
        .rename(columns={'YEAR': 'year', 'MONTH': 'month', 'DAY': 'day', 'HOUR': 'hour', 'MINUTE': 'minute'}),
    errors='coerce'
)

data = data.sort_values('incident_datetime').reset_index(drop=True)

# 1) neighbourhood_total and neighbourhood_share_by_type
neighbourhood_total = data.groupby('NEIGHBOURHOOD')['TYPE'].transform('size')
neighbourhood_type_count = data.groupby(['NEIGHBOURHOOD', 'TYPE'])['TYPE'].transform('size')

data['neighbourhood_total'] = neighbourhood_total
data['neighbourhood_share_by_type'] = neighbourhood_type_count / neighbourhood_total

# 2) block_repeat_count (same hundred block across full period)
data['block_repeat_count'] = data.groupby('HUNDRED_BLOCK')['HUNDRED_BLOCK'].transform('size')

# 3) type_count_neighbourhood_7d and 30d rolling counts (stable index alignment)
rolling_7d = (
    data.groupby(['NEIGHBOURHOOD', 'TYPE'], group_keys=False)
        .apply(
            lambda group: pd.Series(
                group.set_index('incident_datetime')['HOUR'].rolling('7D').count().values,
                index=group.index
            ),
            include_groups=False
        )
)

rolling_30d = (
    data.groupby(['NEIGHBOURHOOD', 'TYPE'], group_keys=False)
        .apply(
            lambda group: pd.Series(
                group.set_index('incident_datetime')['HOUR'].rolling('30D').count().values,
                index=group.index
            ),
            include_groups=False
        )
)

data['type_count_neighbourhood_7d'] = rolling_7d.astype('Int64')
data['type_count_neighbourhood_30d'] = rolling_30d.astype('Int64')

# 4) pct_change_vs_prev_month by neighbourhood and type
data['year_month'] = data['incident_datetime'].dt.to_period('M')

monthly_counts = (
    data.groupby(['NEIGHBOURHOOD', 'TYPE', 'year_month'])
        .size()
        .rename('monthly_count')
        .reset_index()
        .sort_values(['NEIGHBOURHOOD', 'TYPE', 'year_month'])
)

monthly_counts['pct_change_vs_prev_month'] = (
    monthly_counts.groupby(['NEIGHBOURHOOD', 'TYPE'])['monthly_count'].pct_change()
)

data = data.merge(
    monthly_counts[['NEIGHBOURHOOD', 'TYPE', 'year_month', 'pct_change_vs_prev_month']],
    on=['NEIGHBOURHOOD', 'TYPE', 'year_month'],
    how='left'
)

# 5) days_since_last_same_type_in_neighbourhood
prev_datetime = data.groupby(['NEIGHBOURHOOD', 'TYPE'])['incident_datetime'].shift(1)
data['days_since_last_same_type_in_neighbourhood'] = (
    (data['incident_datetime'] - prev_datetime).dt.total_seconds() / 86400
)

# 6) is_top5_type and is_rare_type
crime_type_counts = data['TYPE'].value_counts(dropna=False)
top5_types = set(crime_type_counts.head(5).index)
rare_cutoff = crime_type_counts.quantile(0.25)
rare_types = set(crime_type_counts[crime_type_counts <= rare_cutoff].index)

data['is_top5_type'] = data['TYPE'].isin(top5_types)
data['is_rare_type'] = data['TYPE'].isin(rare_types)

# Quick check of new fields
new_cols = [
    'neighbourhood_total',
    'neighbourhood_share_by_type',
    'block_repeat_count',
    'type_count_neighbourhood_7d',
    'type_count_neighbourhood_30d',
    'pct_change_vs_prev_month',
    'days_since_last_same_type_in_neighbourhood',
    'is_top5_type',
    'is_rare_type'
]

data[new_cols].head()

C:\Users\austi\AppData\Local\Temp\ipykernel_4624\652901220.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
C:\Users\austi\AppData\Local\Temp\ipykernel_4624\652901220.py:34: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,neighbourhood_total,neighbourhood_share_by_type,block_repeat_count,type_count_neighbourhood_7d,type_count_neighbourhood_30d,pct_change_vs_prev_month,days_since_last_same_type_in_neighbourhood,is_top5_type,is_rare_type
0,57010.0,0.113734,18511,1,1,NaN,NaN,True,False
1,12223.0,0.069786,18511,1,1,NaN,NaN,True,False
2,6191.0,0.069133,18511,1,1,NaN,NaN,True,False
3,6191.0,0.069133,18511,2,2,NaN,0.0,True,False
4,5845.0,0.343370,6,1,1,NaN,NaN,True,False
